# Model supplementary figures S1–S4

Run the setup and table-loading cells before the desired panels. Adjust panel dimensions, font sizes and color scales in the plotting calls.
The MultiNeD figure workbench supplies the model source tables.

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
root = next(path for path in (start, *start.parents) if (path / "supplementary/supp_plotting").is_dir())
sys.path.insert(0, str(root))

from matplotlib.colors import LinearSegmentedColormap

from MultiNeD.figure_work.scripts.figure_data import load_table
from supplementary.supp_plotting.editing import export_panel
from supplementary.supp_plotting.model_dynamics import (
    draw_dense_field, load_dense_field, plot_loto_w1_heatmap, plot_state_density_overview,
)
from supplementary.supp_plotting.model_single_panels import (
    plot_grouped_delta_sensitivity, plot_grouped_treatment_weights,
)
from supplementary.supp_plotting.model_style import apply_global_style
from supplementary.supp_plotting.model_training import plot_loto_day_error, plot_training_history
from supplementary.supp_plotting.paths import MODEL_DIR, figure_output_dir

OUTPUT_DIR = figure_output_dir("model")
apply_global_style()

## Load saved data

In [ ]:
loto = load_table("balanced_loto_predictions.csv", MODEL_DIR)
dynamics_paired = load_table("dynamics_loto_targets.csv", MODEL_DIR)
importance = load_table("program_importance_by_delta.csv", MODEL_DIR)
particles = load_table("downstream_trajectory_particles.csv", MODEL_DIR)
baseline_particles = particles.loc[particles.kind.eq("predicted")]
dense_field = load_dense_field(MODEL_DIR)

## S1a — W1 error on held-out data

In [ ]:
figsize = (11.52 / 2.54, 6.5 / 2.54)
cmap = LinearSegmentedColormap.from_list("soft_blue_red", ("#A8C4D3", "#F6F4F0", "#D5ADA5"))
fig = plot_loto_w1_heatmap(dynamics_paired, figsize=figsize, cmap=cmap, font_size=10)
export_panel(fig, "S1a_cell_loto_wasserstein", output_dir=OUTPUT_DIR)

## S1b — Tumor prediction error by treatment and held-out day

In [ ]:
figsize = (9 / 2.54, 6.5 / 2.54)
cmap = LinearSegmentedColormap.from_list("soft_green_purple", ("#C3D8CC", "#F5F3E8", "#D4C7DD"))
fig = plot_loto_day_error(loto, figsize=figsize, cmap=cmap, font_size=10)
export_panel(fig, "S1b_volume_loto_error", output_dir=OUTPUT_DIR)

## S1c — Drift model training curves

Plot the saved total loss and its components over 800 epochs.

In [ ]:
history = load_table("drift_training_history.csv", MODEL_DIR)
components = [
    ("ot_loss", "Local OT", "#3C78A8"),
    ("rollout_loss", "Rollout OT", "#D97732"),
    ("drift_l2", r"Displacement regularization, $\mathcal{R}_{\mathrm{disp}}$", "#458B74"),
]
fig = plot_training_history(history, components, figsize=(20.5 / 2.54, 7 / 2.54), font_size=10)
export_panel(fig, "S1c_drift_training_loss", output_dir=OUTPUT_DIR)

## S1d — Reaction model training curves

Use the saved records for 600 epochs. The total loss is `z_loss + 2 × volume_loss`. Component curves show unweighted values.

In [ ]:
history = load_table("reaction_training_history.csv", MODEL_DIR)
components = [
    ("z_loss", r"Cellular-distribution loss, $\mathcal{L}_z$", "#3C78A8"),
    ("volume_loss", r"Tumor-volume loss, $\mathcal{L}_C$", "#D97732"),
]
fig = plot_training_history(history, components, figsize=(20.5 / 2.54, 7 / 2.54), font_size=10)
export_panel(fig, "S1d_joint_training_loss", output_dir=OUTPUT_DIR)

## S2a–d — Direct drift fields for four program backgrounds

The four panels share dimensions and color scales. `PROGRAM_VIEWS` defines the programs and export filenames.

In [ ]:
figsize = (10.5 / 2.54, 8 / 2.54)
PROGRAM_VIEWS = {
    "Precursor": "S2a_drift_precursor",
    "Activation": "S2b_drift_activation",
    "Early effector": "S2c_drift_early_effector",
    "Dysfunction": "S2d_drift_dysfunction",
}
for program, stem in PROGRAM_VIEWS.items():
    fig = draw_dense_field(dense_field, program, figsize=figsize, font_size=10, tick_size=9, cmap="RdBu_r")
    export_panel(fig, stem, output_dir=OUTPUT_DIR)

## S3 — State densities for four treatments and five programs

The bundled table has 128 equal-weight particles per treatment at each of 121 time points. Regenerated trajectories use the same time grid and contribute to the density according to their saved particle weights.

In [ ]:
fig = plot_state_density_overview(
    baseline_particles, figsize=(21 / 2.54, 14.5 / 2.54), cmap="viridis", font_size=10,
)
export_panel(fig, "S3_state_density", output_dir=OUTPUT_DIR)

## S4a — Treatment-specific program weights

Each of the five programs has four treatment bars showing the saved mean ± SD at δ = 0.5. Each bar summarizes 18 day–seed records.

In [ ]:
fig = plot_grouped_treatment_weights(importance, figsize=(10.5 / 2.54, 7 / 2.54), font_size=10)
export_panel(fig, "S4a_treatment_program_weights", output_dir=OUTPUT_DIR)

## S4b — Sensitivity to perturbation magnitude

The horizontal axis groups the five program bars by treatment. The discrete colorbar on the right identifies the programs.
Bars show the saved means at δ = 0.5. Circles, squares and triangles show the saved means at δ = 0.25, 0.5 and 1, respectively.

In [ ]:
fig = plot_grouped_delta_sensitivity(importance, figsize=(10.5 / 2.54, 7 / 2.54), font_size=10)
export_panel(fig, "S4b_perturbation_sensitivity", output_dir=OUTPUT_DIR)